Another version of plotting_firing_rate_from_database_to plot normalised FIRST peaks

In [1]:
import pyabf
from scipy.signal import find_peaks
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob 


def analyze_by_condition(column_to_use="normalized_first_peak", results_csv_path="/Users/rbondare/ephys/ephys_results/all_peak_results_final.csv"):
    """
    Analyze ABF files grouped by cell ID and condition, using a specific column from the results CSV file.
    
    Parameters:
    -----------
    column_to_use : str, optional
        Column name from the results CSV file to use for further processing
    results_csv_path : str, optional
        Path to the results CSV file
    
    Returns:
    --------
    DataFrame
        Results grouped by cell ID and condition for the chosen column
    """
    if results_csv_path is None or not os.path.exists(results_csv_path):
        raise FileNotFoundError(f"Results CSV file not found at {results_csv_path}")
    df = pd.read_csv(results_csv_path, delimiter=';')

    print("Columns in loaded DataFrame:", df.columns.tolist())  # Debug: show columns
    # Remove rows marked with "exclude" in the comment column
    if 'comment' not in df.columns:
        raise KeyError("Column 'comment' not found in the CSV file. Available columns: " + str(df.columns.tolist()))
    df_filtered = df[df['comment'] != 'exclude'].copy()

    grouped_results = []
    
    for cell_idx, cell_group in df_filtered.groupby('ID'):
        genotype = cell_group['genotype'].iloc[0]
        cell_results = {"ID": cell_idx, "genotype": genotype}
        
        for condition in ['baseline', 'Noradrenaline', 'wash']:  
            condition_data = cell_group[cell_group['condition'] == condition]
            if condition_data.empty:
                cell_results[condition] = None
            else:
                # Get the value from the filtered dataframe
                file_name = condition_data['filename'].iloc[0]
                matching_rows = df_filtered[
                    (df_filtered['filename'] == file_name) & 
                    (df_filtered['condition'] == condition) & 
                    (df_filtered['ID'] == cell_idx)
                ]
                
                if not matching_rows.empty and column_to_use in matching_rows.columns:
                    cell_results[condition] = matching_rows[column_to_use].iloc[0]
                else:
                    cell_results[condition] = None
        
        grouped_results.append(cell_results)
    
    result_df = pd.DataFrame(grouped_results)
    return result_df


def plot_by_genotype(results_df):
    """
    Plot spike probability (or chosen metric) by condition for each genotype.
    Each cell is linked across conditions with a line. One figure per genotype.
    
    Parameters:
    -----------
    results_df : DataFrame
        DataFrame with columns: ID, genotype, baseline, Noradrenaline, wash
    """
    genotypes = results_df['genotype'].unique()
    conditions = ['baseline', 'Noradrenaline', 'wash']
    colors = ['lightgrey', 'lightcoral', 'lightblue']
    
    for genotype in genotypes:
        group = results_df[results_df['genotype'] == genotype]
        if group.empty:
            continue
        
        plt.figure(figsize=(4, 6))
        # Plot each cell as a line across conditions
        for idx, row in group.iterrows():
            y = [row[cond] for cond in conditions]
            # Only plot if we have at least some data points
            if any(val is not None and not pd.isna(val) for val in y):
                plt.plot(range(len(conditions)), y, color='lightgray', alpha=0.8, linewidth=1)
                plt.scatter(range(len(conditions)), y, color=colors, s=60, zorder=3)
        
        # Plot mean as thick black bar for each condition
        for i, cond in enumerate(conditions):
            y_vals = group[cond].dropna().values
            if len(y_vals) > 0:
                plt.hlines(np.mean(y_vals), i - 0.15, i + 0.15, color='black', linewidth=2.5)
        
        plt.xticks(range(len(conditions)), ['baseline', 'NA', 'wash'], rotation=45)
        plt.ylabel("Probability of Spike per Sweep")
        plt.title(f"{genotype}", fontsize=12)
        plt.ylim(-0.05, 1.05)
        plt.tight_layout()
        plt.show()


if __name__ == "__main__":
    # Process the CSV file
    csv_path = "/Users/rbondare/ephys/ephys_results/all_peak_results_final.csv"  # Full path
    
    # Process with normalized_all_peaks column
    results_df = analyze_by_condition(column_to_use='normalized_first_peak', results_csv_path=csv_path)
    
    # Show first few rows
    print("\nFirst 5 rows of processed data:")
    print(results_df.head())
    
    # Plot data by genotype
    plot_by_genotype(results_df)
    
    # Analyze data completeness
    print("\nData completeness summary:")
    for genotype in results_df['genotype'].unique():
        genotype_data = results_df[results_df['genotype'] == genotype]
        print(f"\n{genotype}:")
        print(f"  Total cells: {len(genotype_data)}")
        for condition in ['baseline', 'Noradrenaline', 'wash']:
            non_null_count = genotype_data[condition].count()
            print(f"  {condition}: {non_null_count} cells")
    
    # Show cells with complete data for all conditions (good for matched analysis)
    complete_data = results_df.dropna(subset=['baseline', 'Noradrenaline', 'wash'])
    print(f"\nCells with complete data for all conditions: {len(complete_data)}")
    print("By genotype:")
    print(complete_data['genotype'].value_counts())

FileNotFoundError: Results CSV file not found at /Users/rbondare/ephys/ephys_results/all_peak_results_final.csv